# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdullahIdrees291/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions


### Finding 1

The research paper reports that AI-driven search results are associated with changes in search visibility and organic performance.

**My methodology question:** Where does the label or outcome measure come from, and how was it measured? I would want to understand whether the reported outcome comes from observed search data, a defined benchmark, or another source. This would help confirm that the label is measured consistently across the examples.

### Finding 2

The research paper reports differences in performance between content influenced by AI-driven search and traditional search.

**My methodology question:** Does the validation design support the strength of this claim? In particular, I would want to know whether the comparison controls for differences between clients, content types, and other factors that could affect performance. A grouped or time-aware validation design could help show whether the observed pattern remains consistent on unseen data.

These questions are intended as constructive checks on how the findings were measured and validated. They do not reject the findings; they identify what evidence would make the claims easier to interpret and reproduce.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

### Before: Random split

In Week 5, the Logistic Regression model was evaluated using a random stratified 80/20 split. The measured results were 0.8222 accuracy, 0.6683 F1, and 0.9132 ROC-AUC.

### After: Client-grouped split

For this audit, I will use a client-grouped split so that records from the same client are not present in both training and test sets. This provides a more honest test of whether the observed model performance transfers to unseen clients.

I will compare the before and after results using the same evaluation metrics. Any difference will be reported as an observed change in measured performance, not as proof of real-world future performance.


In [3]:
!git clone https://github.com/AbdullahIdrees291/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 167 (delta 66), reused 95 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 1.97 MiB | 11.07 MiB/s, done.
Resolving deltas: 100% (66/66), done.


In [4]:
import os

os.chdir("/content/flyrank-ml-internship")

print("Current folder:", os.getcwd())

Current folder: /content/flyrank-ml-internship


In [5]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target
model_df = df.copy()
model_df["positive_trend"] = (model_df["trend_pct"] > 0).astype(int)

# Remove target, outcome and identifiers
drop_cols = [
    "positive_trend",
    "trend_pct",
    "trend_direction",
    "content_id",
    "client_id"
]

X_model = model_df.drop(columns=drop_cols)
y_model = model_df["positive_trend"]

# Client ID is used only to create groups
groups = model_df["client_id"]

# Client-grouped split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_model, y_model, groups=groups)
)

X_train = X_model.iloc[train_idx]
X_test = X_model.iloc[test_idx]
y_train = y_model.iloc[train_idx]
y_test = y_model.iloc[test_idx]

# Check client separation
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Overlapping clients:", len(train_clients.intersection(test_clients)))

# Identify feature types
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Logistic Regression
grouped_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train
grouped_model.fit(X_train, y_train)

# Predictions
y_pred = grouped_model.predict(X_test)
y_prob = grouped_model.predict_proba(X_test)[:, 1]

# Metrics
after_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print("\nCLIENT-GROUPED RESULTS")

for metric, value in after_metrics.items():
    print(f"{metric}: {value:.4f}")

# Week-5 random split results
before_metrics = {
    "Accuracy": 0.8222,
    "Precision": 0.5582,
    "Recall": 0.8327,
    "F1": 0.6683,
    "ROC-AUC": 0.9132
}

# Before vs After
comparison = pd.DataFrame([
    {"Split": "Before - Random", **before_metrics},
    {"Split": "After - Client Grouped", **after_metrics}
])

print("\nBEFORE VS AFTER")
display(comparison.round(4))

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Overlapping clients: 0

CLIENT-GROUPED RESULTS
Accuracy: 0.7240
Precision: 0.4521
Recall: 0.7461
F1: 0.5631
ROC-AUC: 0.8166

BEFORE VS AFTER


,Split,Accuracy,Precision,Recall,F1,ROC-AUC
0,Before - Random,0.8222,0.5582,0.8327,0.6683,0.9132
1,After - Client Grouped,0.7240,0.4521,0.7461,0.5631,0.8166


## 3. Leakage audit

I reviewed the final feature set for possible target leakage. The target is `positive_trend`, which is created from `trend_pct`. Therefore, `trend_pct` must not be used as a feature.

I also excluded `trend_direction`, because it directly describes the trend outcome and could reveal the target.

`content_id` and `client_id` were excluded from the model features. `client_id` was used only for grouped validation so that the same client does not appear in both training and test sets.

The remaining features are treated as observed content, search, traffic, engagement, and freshness signals. I did not treat the model results as evidence that changing these features causes a positive trend.

Overall, the audit found clear outcome-related columns that needed to be excluded. The final model feature set excludes these columns, while `client_id` is retained only for grouping the validation split.


In [7]:
# Section 3: Leakage audit

target_col = "positive_trend"

# Columns that could directly reveal the target
leakage_candidates = [
    "trend_pct",
    "trend_direction"
]

# Columns excluded because they are identifiers
identifier_columns = [
    "content_id",
    "client_id"
]

print("TARGET:", target_col)

print("\nPotential outcome/leakage columns:")
for col in leakage_candidates:
    print(f"{col}: present in dataset = {col in df.columns}, used as model feature = {col in X_model.columns}")

print("\nIdentifier columns:")
for col in identifier_columns:
    print(f"{col}: present in dataset = {col in df.columns}, used as model feature = {col in X_model.columns}")

print("\nFinal feature count:", X_model.shape[1])

# Direct leakage check
direct_leakage_in_features = [
    col for col in leakage_candidates
    if col in X_model.columns
]

print("\nDirect leakage columns remaining in final features:")
print(direct_leakage_in_features)

if len(direct_leakage_in_features) == 0:
    print("PASS: No direct outcome columns remain in the final feature set.")
else:
    print("WARNING: Review these columns before submission.")

TARGET: positive_trend

Potential outcome/leakage columns:
trend_pct: present in dataset = True, used as model feature = False
trend_direction: present in dataset = True, used as model feature = False

Identifier columns:
content_id: present in dataset = True, used as model feature = False
client_id: present in dataset = True, used as model feature = False

Final feature count: 40

Direct leakage columns remaining in final features:
[]
PASS: No direct outcome columns remain in the final feature set.


## 4. Claim rewrite

### Original claim

The Logistic Regression model substantially outperformed the Week-4 baseline and can identify content that will have a positive trend.

### Safer claim

On the random test split, the Logistic Regression model measured higher accuracy, F1, and ROC-AUC than the Week-4 baseline. Under client-grouped validation, measured performance was lower but remained directionally useful for identifying positive-trend cases.

These results provide decision-support evidence that the available signals contain predictive information. They do not establish that the model will predict future performance reliably for every client, or that changing content will cause a positive trend.


In [8]:
# Section 4: Claim rewrite - measured comparison

print("Original Week-5 F1:", before_metrics["F1"])
print("Client-grouped F1:", after_metrics["F1"])

print("\nOriginal Week-5 ROC-AUC:", before_metrics["ROC-AUC"])
print("Client-grouped ROC-AUC:", after_metrics["ROC-AUC"])

print("\nSafe interpretation:")
print(
    "Measured performance was lower under client-grouped validation. "
    "The results are directional decision-support evidence, "
    "not proof of future performance or causation."
)

Original Week-5 F1: 0.6683
Client-grouped F1: 0.5630619059851014

Original Week-5 ROC-AUC: 0.9132
Client-grouped ROC-AUC: 0.8166326492432876

Safe interpretation:
Measured performance was lower under client-grouped validation. The results are directional decision-support evidence, not proof of future performance or causation.


## Self-check

* [x] Section 1 contains two research-paper findings and constructive methodology questions.
* [x] Section 2 compares the original random split with a client-grouped split.
* [x] The grouped split has zero overlapping clients between training and test sets.
* [x] Section 3 includes a leakage audit of the final feature set.
* [x] `trend_pct` and `trend_direction` are excluded from the model features.
* [x] `content_id` and `client_id` are excluded from the model features.
* [x] Section 4 rewrites the original claim using measured, directional, and decision-support language.
* [x] The notebook contains no client names, private queries, or sensitive information.
* [ ] Runtime → Run all completes without errors.
* [ ] The completed notebook is saved and committed under `work/notebooks/w06_validation_audit.ipynb`.
